In [1]:
import pandas as pd

# Load data
df = pd.read_csv("abs_tech.csv")

# Identify the starting column
start_col = "EngagementSurvey"

# Get all columns from EngagementSurvey onward
cols = df.columns[df.columns.get_loc(start_col):]

# Remove 'Remote' because it is not numeric
cols = [c for c in cols if c != "Remote"]

# Keep only numeric columns
numeric_cols = df[cols].select_dtypes(include="number").columns.tolist()

# Create two groups:
# 1. Active
# 2. Voluntarily Terminated + Terminated (merged)
df["EmpGroup"] = df["EmploymentStatus"].apply(
    lambda x: "Active" if x == "Active" else
              "Vol/Terminated" if x in ["Voluntarily Terminated", "Terminated"] else None
)

# Filter only the two groups
df_filtered = df[df["EmpGroup"].isin(["Active", "Vol/Terminated"])]

# Function to compute metrics
def compute_metrics(group_df):
    return pd.DataFrame({
        "Mean": group_df[numeric_cols].mean(),
        "StdDev": group_df[numeric_cols].std(),
        "Median": group_df[numeric_cols].median(),
        "Min": group_df[numeric_cols].min(),
        "Max": group_df[numeric_cols].max()
    })

# Compute metrics for each group
active_stats = compute_metrics(df_filtered[df_filtered["EmpGroup"] == "Active"])
terminated_stats = compute_metrics(df_filtered[df_filtered["EmpGroup"] == "Vol/Terminated"])

# Add group labels
active_stats["Group"] = "Active"
terminated_stats["Group"] = "Vol/Terminated"

# Combine into one table
final_table = pd.concat([active_stats, terminated_stats], axis=0)

# Reorder columns
final_table = final_table[["Group", "Mean", "StdDev", "Median", "Min", "Max"]]

print(final_table)

                               Group       Mean     StdDev  Median   Min  \
EngagementSurvey              Active   4.119807   0.781390    4.29  1.12   
EmpSatisfaction               Active   3.893720   0.933887    4.00  1.00   
SpecialProjectsCount          Active   1.463768   2.532774    0.00  0.00   
DaysLateLast30                Active   0.289855   1.058045    0.00  0.00   
Absences                      Active   9.830918   5.846367   10.00  1.00   
ManPos                        Active   0.289855   0.454795    0.00  0.00   
TechLev                       Active   5.222222   1.962980    5.00  1.00   
JobStr                        Active   3.091787   0.873716    3.00  1.00   
ProjColl                      Active   3.135266   0.898252    3.00  1.00   
ProjSelf                      Active   3.342995   0.888566    3.00  1.00   
ProjLead                      Active   3.178744   0.971403    3.00  1.00   
TeamIden                      Active   3.458937   1.027300    4.00  1.00   
OrgIden     

In [5]:
import pandas as pd

# Assuming final_table already exists from previous steps

# Split into two groups
active = final_table[final_table["Group"] == "Active"].drop(columns="Group")
terminated = final_table[final_table["Group"] == "Vol/Terminated"].drop(columns="Group")

# Ensure aligned indices
active = active.sort_index()
terminated = terminated.sort_index()

# Compute absolute differences
diff = active - terminated

# Compute percentage differences
pct_diff = (diff / terminated) * 100

# Format percentage differences as strings with %
pct_diff = pct_diff.applymap(lambda x: f"{x:.2f}%" if pd.notnull(x) else None)

# Combine absolute and percentage differences side-by-side
combined = pd.DataFrame()

for col in diff.columns:
    combined[f"{col}_Diff"] = diff[col]
    combined[f"{col}_PctDiff"] = pct_diff[col]

print(combined)

                      Mean_Diff Mean_PctDiff  StdDev_Diff StdDev_PctDiff  \
AIConf                 0.610562       38.93%     0.252624         30.52%   
AIUse                  0.480567       22.74%     0.103868         11.35%   
Absences              -1.123628      -10.26%     0.051187          0.88%   
CarOpp                 1.035189       56.58%     0.165571         18.67%   
DaysLateLast30        -0.119236      -29.15%    -0.260578        -19.76%   
EmpSatisfaction       -0.004007       -0.10%     0.075664          8.82%   
EngagementSurvey      -0.060421       -1.45%     0.071027         10.00%   
Feedback               0.690437       26.08%     0.002019          0.18%   
InnoCont               0.313351       11.68%     0.079444          9.81%   
JobStr                 0.125878        4.24%    -0.181636        -17.21%   
ManPos                 0.153491      112.56%     0.109654         31.77%   
Network                0.459102       17.80%     0.092861         12.32%   
OrgIden     

C:\Users\ignas\AppData\Local\Temp\ipykernel_31372\1656274037.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pct_diff = pct_diff.applymap(lambda x: f"{x:.2f}%" if pd.notnull(x) else None)


In [6]:
combined.to_csv("perc_diff_employment_status.csv")